<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Distributions in practice

**[C03-M03-L03 · Distributions in practice](https://learning.nextia-ai.com/courses/math/m03/distributions-in-practice/)** · C03, Essential Mathematics and Statistics for AI · Module 3, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** calculate the binomial chances of urgent tickets in a batch, plan the senior agent's hour, see where the normal curve fits (means of many values) and where it does not (skewed reply times), and do the module practice: how a rare event changes the meaning of a flag.

| | |
|---|---|
| **Time** | About 30 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | The frequency tree for 10,000 tickets, from [Conditional probability: Count 10,000 tickets](https://learning.nextia-ai.com/courses/math/m03/conditional-probability/#count-10000-tickets). Independence, from [Events and uncertainty](https://learning.nextia-ai.com/courses/math/m03/events-and-uncertainty/#independence-does-one-event-change-the-other). The mean and the SD, from [Describe a dataset: Spread](https://learning.nextia-ai.com/courses/math/m01/describe-a-dataset/#spread-range-variance-and-standard-deviation). |
| **Tested** | 2026-10-07, Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m03/distributions-in-practice/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M03-L03-distributions-in-practice/distributions-in-practice-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

## Setup: the simulated reply times

The next cell defines `times`: the first-reply times, in minutes, of 400 simulated tickets, as in the lesson's second chart. A program made them with a fixed seed, so they are always the same. The data is fictional.

Run the cell. You should see `Ready: 400 reply times, 10540 minutes in total`.

In [ ]:
# The data, written in full (you do not need to read it).
times = [
    11, 12, 7, 18, 11, 34, 17, 37, 22, 15, 40, 11, 33, 26, 16, 35, 61, 31, 13, 12,
    47, 38, 41, 26, 11, 14, 53, 53, 15, 5, 15, 23, 24, 14, 31, 30, 13, 19, 39, 10,
    39, 11, 14, 24, 15, 30, 9, 41, 13, 77, 12, 11, 20, 55, 43, 19, 9, 9, 13, 22,
    9, 26, 16, 18, 29, 63, 62, 34, 18, 54, 11, 26, 28, 5, 37, 87, 25, 12, 12, 28,
    21, 21, 17, 13, 14, 23, 16, 14, 24, 27, 26, 18, 21, 56, 23, 32, 36, 10, 10, 18,
    24, 26, 18, 21, 65, 39, 19, 23, 69, 7, 91, 13, 39, 32, 29, 12, 34, 26, 36, 14,
    18, 30, 11, 8, 11, 14, 14, 13, 24, 53, 23, 25, 26, 12, 12, 56, 20, 27, 17, 29,
    56, 9, 10, 64, 25, 20, 56, 46, 24, 42, 47, 26, 25, 18, 18, 24, 52, 30, 13, 34,
    18, 17, 30, 19, 35, 8, 18, 28, 28, 38, 39, 17, 52, 13, 17, 52, 35, 27, 29, 29,
    26, 62, 41, 26, 45, 27, 13, 25, 24, 35, 32, 24, 10, 7, 16, 15, 19, 13, 42, 16,
    20, 19, 27, 32, 6, 26, 25, 17, 55, 31, 13, 17, 30, 35, 16, 23, 14, 58, 16, 21,
    31, 22, 19, 24, 36, 29, 14, 39, 19, 18, 38, 24, 23, 16, 27, 22, 23, 45, 18, 9,
    33, 9, 20, 14, 65, 22, 40, 10, 19, 12, 7, 57, 16, 15, 36, 30, 17, 13, 14, 49,
    25, 51, 21, 35, 16, 23, 24, 33, 25, 28, 11, 21, 20, 22, 20, 28, 38, 42, 16, 21,
    35, 16, 27, 50, 33, 14, 24, 6, 35, 10, 32, 20, 17, 33, 56, 15, 30, 30, 44, 13,
    9, 44, 22, 23, 23, 18, 22, 17, 17, 15, 70, 27, 12, 28, 34, 13, 32, 25, 36, 27,
    20, 13, 19, 41, 42, 12, 20, 45, 30, 6, 26, 14, 64, 16, 10, 5, 12, 29, 44, 27,
    19, 12, 14, 11, 25, 17, 57, 3, 38, 22, 39, 18, 14, 10, 34, 22, 13, 40, 32, 22,
    21, 55, 18, 29, 37, 15, 10, 20, 22, 20, 19, 60, 39, 40, 105, 29, 11, 60, 10, 17,
    32, 12, 9, 17, 7, 43, 48, 32, 15, 10, 33, 33, 27, 46, 51, 23, 32, 29, 30, 53,
]
print(f"Ready: {len(times)} reply times, {sum(times)} minutes in total")

## Setup: the sample means

The next cell defines `sample_means`: the mean reply time of each of 300 simulated samples of 30 tickets, as in the lesson's first chart.

Run the cell. You should see `Ready: 300 sample means, from 18.6 to 40.3 minutes`.

In [ ]:
# The data, written in full (you do not need to read it).
sample_means = [
    32.4, 31.8, 27, 25.4, 25.2, 27.7, 22.7, 31.3, 27.1, 27.2, 29.7, 34.1, 23.4, 36.8, 22.1,
    28.4, 29.1, 27.1, 28.6, 36.9, 27.1, 31, 26.8, 26.5, 32.5, 26.3, 27.7, 29.9, 24.3, 28.7,
    25.8, 27.4, 30.5, 23.4, 22.4, 29.5, 27.2, 28.4, 25.4, 31.3, 29, 27.6, 25.6, 30.1, 25.7,
    34.5, 27.4, 23.8, 32.3, 27.3, 26.8, 24.7, 26.9, 26.9, 40.3, 28.1, 27, 28.6, 20.1, 29,
    24.7, 32.3, 31.3, 29, 26.3, 29.3, 31.8, 23.2, 27.7, 24.1, 27.3, 24.6, 27.1, 28.2, 24.1,
    27.3, 32.1, 26.8, 28.5, 26.3, 26.7, 29.6, 20.9, 24.7, 26.2, 29.8, 28.4, 30.8, 28.7, 32,
    30.4, 26.4, 30.5, 28.8, 30.1, 27.9, 26.1, 28.9, 30.2, 24, 35.5, 29.2, 24.9, 30, 29.3,
    26.4, 24.5, 24.7, 25.7, 30.4, 22.5, 36, 30.1, 29, 27.6, 26, 26.9, 29.2, 25.7, 31.5,
    28.2, 24.3, 26, 23.5, 31, 24, 30.2, 31.2, 26, 25.8, 29.4, 29, 27.7, 24.2, 26.8,
    32.2, 30.5, 26.9, 32, 31.6, 22.7, 24.4, 29.3, 25.6, 24.2, 25.1, 29.5, 32.1, 29.5, 34,
    30.2, 21.9, 18.6, 33.7, 28.6, 33.1, 33.2, 30.1, 32.7, 27.4, 30.7, 33.4, 33.3, 26.3, 32.2,
    26.4, 27, 26, 24.1, 28.5, 27.9, 30.2, 25.9, 24.7, 22.7, 33.6, 20.7, 30.2, 25.9, 29.1,
    27.3, 25.3, 23.9, 24.9, 29.2, 23.4, 26.2, 26.8, 27.1, 25.2, 31.9, 23.9, 33.3, 29.3, 22.1,
    30.4, 29.5, 24, 28.2, 32.2, 30.2, 24.8, 28, 30, 27.9, 33.9, 22.8, 35.9, 22.3, 28.8,
    38.5, 28.7, 25.4, 25.1, 22.6, 25.7, 30.4, 25.7, 32.8, 30.7, 29.7, 23.2, 24, 30.1, 26.4,
    29.2, 28.4, 25.7, 27.9, 32.7, 27.8, 27.3, 27.6, 26.7, 29.4, 31.5, 34, 29.9, 29.7, 27.8,
    24.7, 27.2, 26.9, 25.8, 27.1, 28.6, 27.3, 31.8, 27.2, 35.5, 28.6, 28.4, 36.3, 24.4, 22.9,
    27.3, 25.3, 24.9, 31.5, 30.9, 29.9, 28.6, 23.9, 27.4, 24.1, 34.4, 27.6, 27.9, 33.2, 28.4,
    26, 29.2, 30.4, 29.4, 27.5, 23.9, 30.1, 26.5, 25.9, 21.9, 25.7, 25.3, 31.3, 32.3, 27.1,
    29.7, 23.9, 27.8, 23.7, 34.2, 28.2, 25.7, 26.9, 29.6, 22.5, 29.9, 26.8, 22.6, 29, 32.4,
]
print(f"Ready: {len(sample_means)} sample means, from {min(sample_means)} to {max(sample_means)} minutes")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Bernoulli: one ticket

A **distribution** lists every possible value and how likely each one is. One ticket is urgent (1) or not urgent (0). At Larkfield, the chance of 1 is p = 0.08. That is a **Bernoulli distribution**. Its mean is p.

The next cell makes 10,000 random tickets with a fixed seed. Each ticket is 1 with a chance of 0.08.

> **Predict.** What is the mean of the 10,000 zeros and ones? Then run the cell.

In [ ]:
random.seed(3)
tickets = [1 if random.random() < 0.08 else 0 for t in range(10_000)]
print("urgent tickets:", sum(tickets), "of", len(tickets))
print("mean of the 0s and 1s:", statistics.mean(tickets))

> **Check.** You should see 777 urgent tickets and a mean of `0.0777`: close to 0.08, but not equal. The mean of zeros and ones is the share of ones.

## 2. Binomial: urgent tickets in a batch of 50

About 50 tickets arrive in an hour. The **binomial distribution** gives the chance of each count of urgent tickets, from 0 to 50. It needs a fixed number of tickets (n = 50), the same chance for each ticket (p = 0.08), and independent tickets.

The next cell defines `chance(n, p, k)`: the chance of exactly k urgent tickets among n. `math.comb(n, k)` counts the ways to choose which k tickets are urgent. The cell works with logarithms, so that it also works for n = 1,200 (the plain product gives numbers too large for Python). The lesson's explorer does the same.

> **Predict.** Which count of urgent tickets is the most likely in a batch of 50? Then run the cell.

In [ ]:
def chance(n, p, k):
    """Chance of exactly k urgent tickets among n, if each one is urgent with chance p."""
    if p in (0, 1):
        return 1.0 if k == n * p else 0.0
    return math.exp(math.log(math.comb(n, k)) + k * math.log(p) + (n - k) * math.log(1 - p))

def binomial(n, p):
    """The chances of 0, 1, 2, ... n urgent tickets."""
    return [chance(n, p, k) for k in range(n + 1)]

batch = binomial(50, 0.08)
percent = [round(c * 100, 1) for c in batch[:10]] + [round(sum(batch[10:]) * 100, 1)]
bars([str(k) for k in range(10)] + ["10+"], percent)
print("all 51 chances together:", round(sum(batch), 6))

> **Check.** You should see the lesson's table: 1.5, 6.7, 14.3, 19.9, 20.4, 16.3, 10.6, 5.8, 2.7, 1.1 and 0.6 for 10 or more. The most likely counts are 3 and 4. All 51 chances add up to 1.

The chance of 0 urgent tickets is 0.92 multiplied by itself 50 times. The mean is n × p, and the SD is the square root of n × p × (1 − p). Run the cell. You should see `0.0155`, a mean of `4.0` and an SD of `1.92`.

In [ ]:
n, p = 50, 0.08
print("0.92 ** 50 =", round(0.92 ** 50, 4))
print("mean =", n * p)
print("SD   =", round(math.sqrt(n * p * (1 - p)), 2))

## 3. Worked example: Priya plans the senior agent's hour

The senior agent can handle 6 urgent tickets an hour. Grace asks how often that is not enough. Priya adds the chances of 0 to 6, and takes the complement.

Run the cell. You should see 89.8% for 6 or fewer, 10.2% for more than 6 and 4.4% for more than 7.

In [ ]:
def more_than(capacity, n, p):
    """The chance of more than `capacity` urgent tickets among n."""
    return 1 - sum(binomial(n, p)[:capacity + 1])

table(["Question", "Result (%)"], [
    ["6 or fewer", (1 - more_than(6, 50, 0.08)) * 100],
    ["More than 6", more_than(6, 50, 0.08) * 100],
    ["More than 7", more_than(7, 50, 0.08) * 100],
], digits=1)

Priya also says: "In an 8-hour day, at least one such hour happens on about 6 days in 10." That is the complement again: 1 − (chance that all 8 hours are fine).

> **Predict.** Is it closer to 1 day in 10, or 6 days in 10? Then run the cell. You should see `0.58`.

In [ ]:
fine_hour = 1 - more_than(6, 50, 0.08)
print("at least one hour with more than 6:", round(1 - fine_hour ** 8, 2))

## 4. Larger batches: the SD grows with the square root of n

> **Predict.** The batch grows from 50 to 500 tickets. Does the SD grow 10 times? Then run the cell.

In [ ]:
rows = []
for n in [50, 500, 1200]:
    rows.append([n, n * 0.08, math.sqrt(n * 0.08 * 0.92)])
table(["Tickets n", "Mean", "SD"], rows)
print()
print("SD at 500 / SD at 50:", round(math.sqrt(500 * 0.08 * 0.92) / math.sqrt(50 * 0.08 * 0.92), 2))

> **Check.** You should see SDs of 1.92, 6.07 and 9.40, and a ratio of `3.16`: the square root of 10, not 10. A larger batch is more predictable compared with its size.

For a week of 1,200 tickets, the 95% rule of the normal curve gives mean ± 2 SD. The explorer finds the exact range from the binomial chances: the counts between the lowest 2.5% and the highest 2.5% of weeks. Run the cell. You should see `77.2 to 114.8` from the rule, and `78 to 115` from the binomial chances.

In [ ]:
n, p = 1200, 0.08
mean, sd = n * p, math.sqrt(n * p * (1 - p))
print(f"mean ± 2 SD: {mean - 2 * sd:.1f} to {mean + 2 * sd:.1f}")

week = binomial(n, p)
running, low, high = 0, None, None
for k, c in enumerate(week):
    running += c
    if low is None and running > 0.025:
        low = k
    if high is None and running >= 0.975:
        high = k
print("exact range:", low, "to", high)

## 5. Normal: where it fits

The **normal distribution** is the symmetric, bell-shaped curve, set by a mean and an SD. About 68% of values are within 1 SD of the mean, and about 95% within 2 SD. Python's standard library has it: `statistics.NormalDist`.

The mean of many independent values is close to normal. `sample_means` has the mean reply time of 300 samples of 30 tickets.

> **Predict.** What share of the 300 sample means is within 2 SD of their mean? Then run the cell.

In [ ]:
m = statistics.mean(sample_means)
s = statistics.stdev(sample_means)
within = len([v for v in sample_means if abs(v - m) <= 2 * s])
print(f"mean {m:.1f}, SD {s:.1f}")
print(f"within 2 SD: {within} of {len(sample_means)} ({within / len(sample_means):.1%})")

> **Check.** You should see a mean of 28.0, an SD of 3.4 and 287 of 300 (95.7%) within 2 SD: close to the 95% of the normal curve.

The next cell draws the sample means as a text histogram with 2-minute bins. Run it. The bars rise and fall almost symmetrically around 28 minutes.

In [ ]:
lefts = list(range(18, 42, 2))
counts = [len([v for v in sample_means if left <= v < left + 2]) for left in lefts]
bars([f"{left}–{left + 2}" for left in lefts], counts)

## 6. Normal: where it does not fit

Monday's 12 tickets from Module 1 have a mean of 28.75 minutes and an SD of about 21.1 minutes (divide by n). The next cell asks a normal curve with this mean and SD two questions.

> **Predict.** What share of replies does the normal curve put below 0 minutes? Then run the cell.

In [ ]:
monday = [12, 15, 18, 18, 20, 22, 24, 25, 27, 31, 38, 95]
curve = statistics.NormalDist(statistics.mean(monday), statistics.pstdev(monday))
print(f"below 0 minutes:   {curve.cdf(0):.1%}")
print(f"95 minutes is {(95 - curve.mean) / curve.stdev:.1f} SD from the mean")
print(f"95 minutes or more: {1 - curve.cdf(95):.2%}")

> **Check.** You should see 8.7%, 3.1 SD and 0.08%. A reply cannot come before the ticket, and 95 minutes happened to 1 ticket in 12, not 1 in 1,200.

Now the 400 simulated reply times. Run the cell. You should see a median of 23, a mean of 26.4 and an SD of 15.2. Then the normal curve predicts 14.2% below 10 minutes against 5.8% in the data, 4.2% below 0 minutes, and 0.02% above 80 minutes against 3 tickets in the data.

In [ ]:
curve = statistics.NormalDist(statistics.mean(times), statistics.stdev(times))
print(f"median {statistics.median(times):g}, mean {curve.mean:.1f}, SD {curve.stdev:.1f}")
table(["Question", "Normal curve", "Data"], [
    ["Below 10 minutes", f"{curve.cdf(10):.1%}", f"{len([v for v in times if v < 10]) / len(times):.1%}"],
    ["Below 0 minutes", f"{curve.cdf(0):.1%}", "0 tickets"],
    ["Above 80 minutes", f"{1 - curve.cdf(80):.2%}", f"{len([v for v in times if v > 80])} tickets"],
])

Run the cell to see the shape: most bars are between 10 and 40 minutes, and a long, thin tail goes past 100 minutes. The normal curve is symmetric, so it gets both ends wrong.

In [ ]:
lefts = list(range(0, 110, 10))
bars([f"{left}–{left + 10}" for left in lefts], [len([v for v in times if left <= v < left + 10]) for left in lefts])

## 7. Guided practice: a quiet hour

On Sunday mornings, about 20 tickets arrive in an hour. The chance that a ticket is urgent is still 0.08.

> **Your turn.** Replace each `...` with a calculation. You can use `chance`, `more_than`, `math.sqrt` and `**`. Then run the check cell.

In [ ]:
mean_20 = ...           # n x p
sd_20 = ...             # the square root of n x p x (1 - p)
none_20 = ...           # the chance of no urgent ticket
over_3_20 = ...         # the chance of more than 3, in percent

Run the check cell.

In [ ]:
expect("the mean", mean_20, 1.6)
expect("the SD", sd_20, 1.21)
expect("the chance of no urgent ticket", none_20, 0.189, tol=0.001)
expect("the chance of more than 3 (%)", over_3_20, 7.1, tol=0.05)

n × p = 1.6 is far below 5, so the normal curve fits badly here: the counts pile up at 0, 1 and 2, and a normal curve with this mean and SD spills below 0. Write your sentence in `c03-notes.md`.

## 8. Independent variation: an outage hour

Change one condition: during a payment outage, 1 ticket in 5 is urgent (p = 0.20). The batch is still 50 tickets.

> **Your turn.** Calculate the mean, the SD, and the chances (in percent) of more than 6, more than 14 and more than 15 urgent tickets. Then run the check cell.

In [ ]:
mean_outage = ...
sd_outage = ...
over_6_outage = ...      # in percent
over_14_outage = ...     # in percent
over_15_outage = ...     # in percent

Run the check cell.

In [ ]:
expect("the mean", mean_outage, 10)
expect("the SD", sd_outage, 2.83)
expect("the chance of more than 6 (%)", over_6_outage, 89.7, tol=0.05)
expect("the chance of more than 14 (%)", over_14_outage, 6.1, tol=0.05)
expect("the chance of more than 15 (%)", over_15_outage, 3.1, tol=0.05)

A capacity of 6 is not enough in almost every outage hour. A capacity of 14 or 15 keeps the overflow near 5%. In a real outage, tickets are not independent, so the real counts can be even higher than these binomial numbers.

## 9. Failure case: the normal curve for skewed times

A dashboard promises "99.9% of tickets get a reply within mean + 3 SD". For Monday's tickets, that is about 92 minutes.

> **Predict.** How many of Monday's 12 tickets wait longer than mean + 3 SD? Then run the cell.

In [ ]:
limit = statistics.mean(monday) + 3 * statistics.pstdev(monday)
print(f"mean + 3 SD: {limit:.0f} minutes")
print("tickets above it:", len([v for v in monday if v > limit]), "of", len(monday))
print("p95 of the 400 simulated times:", statistics.quantiles(times, n=20, method="inclusive")[18], "minutes")

> **Check.** You should see 92 minutes and 1 of 12 tickets above it (T-2052): about 1 in 12, not 1 in 1,000. The last line shows the **fix**: read a percentile from the data. 95% of the 400 simulated tickets got a reply within 56 minutes.

> **Warning.** Use the normal curve for means of many values and for counts with a large n × p. For skewed measurements, use the median and percentiles.

## 10. Module practice: a rare event changes the meaning of a flag

### Worked example: the urgent flag

The urgency model flags 90 of 100 urgent tickets and 5 of 100 other tickets. Urgent tickets are 8% of all tickets. The next cell counts 10,000 tickets, as in the lesson's frequency tree. Run it. You should see 800, 9200, 720, 80, 460, 8740 and 1180, and a share of `0.610`.

In [ ]:
def flag_counts(base_rate, recall=0.90, false_alarm_rate=0.05, n=10_000):
    """Count 10,000 tickets: every box of the frequency tree, and the share of flags that are real."""
    cases = round(n * base_rate)
    others = n - cases
    real = round(cases * recall)
    false = round(others * false_alarm_rate)
    return {"cases": cases, "others": others, "real flags": real, "missed": cases - real,
            "false alarms": false, "correctly left": others - false, "all flags": real + false,
            "share real": real / (real + false) if real + false else None}

table(["Box", "Count"], [[k, v] for k, v in flag_counts(0.08).items()], digits=3)

Urgent tickets (8%) are 4 times as common as payment problems (2%), so 61% of the flags are real, not 27%. The base rate decides what a flag means.

### Your task: a rarer problem

Chargeback tickets are **1 in 100**. Tomás's flag for them has 90% recall and a 5% false-alarm rate.

> **Your turn.** Count 10,000 tickets step by step, without `flag_counts`. Then write P(flag | chargeback) and P(chargeback | flag). Run the check cell. Write your two sentences for Omar in `c03-notes.md`.

In [ ]:
chargebacks = ...
other_tickets = ...
real_flags = ...
missed = ...
false_alarms = ...
correctly_left = ...
all_flags = ...
p_flag_given_chargeback = ...
p_chargeback_given_flag = ...

Run the check cell.

In [ ]:
expect("chargeback tickets", chargebacks, 100)
expect("other tickets", other_tickets, 9900)
expect("real flags", real_flags, 90)
expect("missed chargebacks", missed, 10)
expect("false alarms", false_alarms, 495)
expect("correctly left", correctly_left, 9405)
expect("all flags", all_flags, 585)
expect("P(flag | chargeback)", p_flag_given_chargeback, 0.90, tol=0.001)
expect("P(chargeback | flag)", p_chargeback_given_flag, 0.154, tol=0.001)

> **Your turn: change one thing.** Run `flag_counts(0.01, false_alarm_rate=0.01)` in a new cell. That is Omar's option: a lower false-alarm rate. You should see 99 false alarms, 189 flags and a share of about 0.476.

The model answer for the sentences is on the lesson page.

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/math/m03/distributions-in-practice/#module-check). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you calculated the binomial chances for a batch of 50 tickets: mean 4, SD 1.92, and more than 6 urgent tickets in 10.2% of hours. The SD grows with the square root of n. The normal curve fits the 300 sample means (95.7% within 2 SD), but not skewed reply times: it puts 8.7% of Monday's replies below 0 minutes. In the module practice, the same flag quality gives 61% real flags at an 8% base rate and only 15.4% at 1%.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Functions and change](https://learning.nextia-ai.com/courses/math/m04/functions-and-change/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m03/distributions-in-practice/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.